# JASA GP rotor-noise — interactive 3D listening

Move the **listener dot** around the drone in 3D, feed in a **rotor-speed signal**,
and hear what the trained Gaussian-process model (Lee et al., *JASA* 159(4):3418,
2026) synthesizes at that point.

**How it works**
- The GP (`src/experiments/gp_rotor_noise/jasa_gp.py`) was trained on the
  CONA-generated `jasa-flyovers` set: a NASA-1Pax quadrotor flyover captured by a
  ground mic grid, speeds `V∈{6,8,10} m/s` (held-out `V∈{7,9}`).
- It predicts, at any `(x, y, V)`, the **Fourier coefficients** of the tonal
  rotor-noise comb (harmonics of the blade-passing frequency) plus a broadband
  floor `σ_b`.
- Your **rotor-speed signal** `Ω(t)` [rev/s] sets the *instantaneous* comb
  frequency (`BPF = 3·Ω`), frequency-modulating the harmonics — a drone spooling
  up/down. `V` and position set the harmonic **amplitudes / directivity**.
- Lifting the dot off the ground applies a `1/r` distance correction relative to
  the trained ground field (a transparent extrapolation for elevation).

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import ipywidgets as W
from IPython.display import display, Audio

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "src" / "experiments").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from experiments.gp_rotor_noise import jasa_gp as J

CKPT = next((p for p in [ROOT / "results/jasa_gp/best.pt",
                         ROOT / "results/jasa_gp/smoke.pt"] if p.exists()), None)
assert CKPT is not None, "Train first: python -m experiments.gp_rotor_noise.train_jasa_gp"
model = J.JasaGPModel.load(CKPT)
FS = J.FS
HOVER_RPS = J.BPF_HZ / J.N_BLADES
DRONE = np.array([0.0, 0.0, 30.0])
print(f"loaded {CKPT.relative_to(ROOT)}  (H={model.cfg.n_harm} harmonics, "
      f"trained speeds {model.cfg.train_speeds})")

In [ ]:
def make_rps(duration, base_rps, mode, depth):
    """Rotor-speed signal Ω(t) [rev/s] for the clip."""
    n = int(round(duration * FS)); t = np.linspace(0.0, duration, n)
    if mode == "constant":     r = np.full(n, base_rps)
    elif mode == "spool up":   r = base_rps * (1.0 + depth * (t / duration))
    elif mode == "spool down": r = base_rps * (1.0 + depth * (1.0 - t / duration))
    elif mode == "wobble":     r = base_rps * (1.0 + depth * np.sin(2 * np.pi * 2.0 * t))
    else:                      r = np.full(n, base_rps)
    return r

def synth_at(pos, V, duration, base_rps, mode, depth, broadband):
    """Synthesize the pressure the listener at world `pos` hears."""
    pos = np.asarray(pos, float)
    x, y = pos[0], abs(pos[1])
    rps = make_rps(duration, base_rps, mode, depth)
    sig = model.synthesize(x, y, float(V), duration=duration, rps=rps, broadband=broadband)
    d_lis = np.linalg.norm(pos - DRONE) + 1e-6
    d_gnd = np.linalg.norm(np.array([x, y, 0.0]) - DRONE) + 1e-6
    return (sig * (d_gnd / d_lis)).astype(np.float32)

In [ ]:
gx = np.arange(J.AFT_X_RANGE[0], J.AFT_X_RANGE[1] + 1, 10.0)
gy = np.arange(J.AFT_Y_RANGE[0], J.AFT_Y_RANGE[1] + 1, 10.0)
GX, GY = np.meshgrid(gx, gy)
scene = go.FigureWidget(
    data=[
        go.Scatter3d(x=GX.ravel(), y=GY.ravel(), z=np.zeros(GX.size), mode="markers",
                     marker=dict(size=2, color="lightgray"), name="training mics"),
        go.Scatter3d(x=[DRONE[0]], y=[DRONE[1]], z=[DRONE[2]], mode="markers",
                     marker=dict(size=7, color="crimson", symbol="diamond"), name="drone"),
        go.Scatter3d(x=[-60.0], y=[0.0], z=[0.0], mode="markers",
                     marker=dict(size=6, color="royalblue"), name="listener"),
        go.Scatter3d(x=[DRONE[0], -60.0], y=[DRONE[1], 0.0], z=[DRONE[2], 0.0], mode="lines",
                     line=dict(color="royalblue", width=2, dash="dot"), name="line of sight"),
    ],
    layout=go.Layout(height=430, margin=dict(l=0, r=0, t=25, b=0),
        scene=dict(xaxis_title="x [m]", yaxis_title="y [m]", zaxis_title="z [m]", aspectmode="data"),
        legend=dict(orientation="h", y=1.02)),
)
def _update_dot(x, y, z):
    with scene.batch_update():
        scene.data[2].x = [x]; scene.data[2].y = [y]; scene.data[2].z = [z]
        scene.data[3].x = [DRONE[0], x]; scene.data[3].y = [DRONE[1], y]; scene.data[3].z = [DRONE[2], z]

In [ ]:
sl = dict(continuous_update=False, style={"description_width": "90px"}, layout=W.Layout(width="320px"))
x_w = W.FloatSlider(value=-60, min=-150, max=20, step=5, description="listener x", **sl)
y_w = W.FloatSlider(value=0, min=-70, max=70, step=5, description="listener y", **sl)
z_w = W.FloatSlider(value=0, min=0, max=60, step=2, description="listener z", **sl)
V_w = W.FloatSlider(value=8, min=4, max=10, step=0.5, description="flight V", **sl)
rps_w = W.FloatSlider(value=HOVER_RPS, min=6, max=20, step=0.2, description="rotor rev/s", readout_format=".1f", **sl)
mode_w = W.Dropdown(options=["constant", "spool up", "spool down", "wobble"], value="constant", description="rps mode", **sl)
depth_w = W.FloatSlider(value=0.15, min=0.0, max=0.6, step=0.02, description="rps depth", **sl)
dur_w = W.FloatSlider(value=1.5, min=0.5, max=4.0, step=0.5, description="duration s", **sl)
bb_w = W.Dropdown(options=["colored", "white", "none"], value="colored", description="broadband", **sl)
go_btn = W.Button(description="▶ Generate & listen", button_style="success", layout=W.Layout(width="320px"))
audio_out = W.Output(); plot_out = W.Output()

for w in (x_w, y_w, z_w):
    w.observe(lambda ch: _update_dot(x_w.value, y_w.value, z_w.value), names="value")
_update_dot(x_w.value, y_w.value, z_w.value)

def _on_go(_):
    pos = [x_w.value, y_w.value, z_w.value]
    sig = synth_at(pos, V_w.value, dur_w.value, rps_w.value, mode_w.value, depth_w.value, bb_w.value)
    with audio_out:
        audio_out.clear_output(wait=True); display(Audio(sig, rate=int(FS), normalize=True))
    with plot_out:
        plot_out.clear_output(wait=True)
        fig, ax = plt.subplots(1, 2, figsize=(11, 3))
        t = np.arange(len(sig)) / FS; m = t <= min(0.25, t[-1])
        ax[0].plot(t[m], sig[m], lw=0.7); ax[0].set(xlabel="t [s]", ylabel="Pa", title="waveform (first 0.25 s)")
        ax[1].specgram(sig, NFFT=2048, Fs=FS, noverlap=1536, cmap="magma")
        ax[1].set(ylim=(0, 1200), xlabel="t [s]", ylabel="Hz", title="spectrogram")
        d = np.linalg.norm(np.array(pos) - DRONE)
        fig.suptitle(f"listener {pos} m  |  {d:.0f} m from drone  |  V={V_w.value} m/s  |  "
                     f"{mode_w.value} @ {rps_w.value:.1f} rev/s", fontsize=9)
        plt.tight_layout(); plt.show()

go_btn.on_click(_on_go)
display(W.HBox([W.VBox([x_w, y_w, z_w, W.HTML("<hr>"), V_w, rps_w, mode_w, depth_w, dur_w, bb_w, W.HTML("<hr>"), go_btn]), scene]))
display(audio_out); display(plot_out)
_on_go(None)

### Notes & caveats
- **Faithful region**: trained on the aft ground grid `x∈[−140,−30], y∈[0,70]` at
  `V∈{6,8,10}`. Far outside it (front of the drone, `V<4`) the posterior reverts
  toward the prior mean — quieter / smoother — the extrapolation behavior the paper
  reports (Fig. 9).
- **Elevation (`z>0`)** is a `1/r` amplitude extrapolation, not a re-trained field
  (all training mics are on the ground).
- **Rotor-speed FM**: trained amplitudes come from a hover-trimmed rotor
  (~11.2 rev/s); driving `Ω(t)` away from that shifts the whole comb (pitch) while
  keeping the learned per-harmonic balance.